# Deep learning I: the curse of dimensionality and training a model

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

All the code of this lecture: a linear regression model written from scratch, its least-squares loss, a search for the parameters with the lowest loss, and the loss surface as a heatmap. There is nothing to type - run the cells in order and compare the output with the lecture.

Later cells use the variables and functions of earlier ones: Runtime > Run all runs everything at once.

## Code-along 1. Plot the prediction and compute the loss

- Load math and plotting libraries:

In [ ]:
# numerical arrays
import numpy as np
# plotting
import matplotlib.pyplot as plt

- Create some input/output data:

In [ ]:
# twelve inputs x and the true outputs y
x = np.array([0.03, 0.19, 0.34, 0.46, 0.78, 0.81, 1.08, 1.18, 1.39, 1.60, 1.65, 1.90])
y = np.array([0.67, 0.85, 1.05, 1.0, 1.40, 1.5, 1.3, 1.54, 1.55, 1.68, 1.73, 1.6 ])

# print both arrays
print(x)
print(y)

- Define a 1D linear regression model:

In [ ]:
def linf(x, phi0, phi1):
    """Linear function y = f(x) = phi0 + phi1 * x
    x: input vector
    phi0: intercept
    phi1: slope
    y: output vector
    """
    y = phi0 + phi1 * x
    return y

- Plotting function:

In [ ]:
def plot(x, y, phi0, phi1):
    """Plot vectors x,y and line f(x)=phi0+phi1*x
    x: x-axis vector (input)
    y: y-axis vector (output)
    """
    # new figure with the data as points
    fig, ax = plt.subplots()
    ax.scatter(x, y)
    # axis ranges and labels
    plt.xlim([0, 2.0])
    plt.ylim([0, 2.0])
    ax.set_xlabel('Input, $x$')
    ax.set_ylabel('Output, $y$')
    # the model's line over the whole x range
    x_line = np.arange(0, 2, 0.01)
    y_line = linf(x_line, phi0, phi1)
    plt.plot(x_line, y_line, 'b-', lw=2)

- Set the intercept and slope and plot the data:

In [ ]:
# a first guess for the two parameters
phi0 = 0.4; phi1 = 0.2
# data and line
plot(x, y, phi0, phi1)

- Function to calculate the loss: the sum of the squared **residuals** -
  the length of the vertical error lines between each of the true
  points {x_i, y_i} and the line.

In [ ]:
def compute_loss(x, y, phi0, phi1):
    # residuals: prediction minus truth, one per point
    loss = phi0 + phi1 * x - y
    # sum of the squared residuals: one number
    loss_scalar = np.sum(loss**2)
    return loss_scalar

- Compute the loss:

In [ ]:
# loss for phi0 = 0.4, phi1 = 0.2
loss = compute_loss(x, y, phi0, phi1)
print(f'Your loss = {loss:3.2f}, ground truth = 7.07')

- Set the intercept and slope to a different value, then plot and
  print the loss:

In [ ]:
# a second guess: falling line
phi0 = 1.60; phi1 = -0.8
plot(x, y, phi0, phi1)

In [ ]:
# loss for phi0 = 1.6, phi1 = -0.8
loss = compute_loss(x, y, phi0, phi1)
print(f'Your loss = {loss:3.2f}, ground truth = 10.28')

## Code-along 2. Search for the minimum of the loss function

- Change the parameters to fit the model: change the intercept until
  the loss won't go lower, then fix the intercept and change the slope
  until the loss is at its minimum.

- The search function:

In [ ]:
def line_search(x, y, phi0, phi1, vary, step=0.5, min_step=1e-4):
    """Hill-climb one parameter ('phi0' or 'phi1'): try a small step up
    and down, keep whichever lowers the loss, and shrink the step
    whenever neither direction helps, until the step is tiny."""

    # loss at the starting point: every step has to beat it
    best_loss = compute_loss(x, y, phi0, phi1)

    # big steps first, then finer and finer
    while step > min_step:
        improved = False
        # try +step, and -step only if +step did not help
        for direction in (1, -1):
            # nudge only the parameter named in 'vary'
            if vary == 'phi0':
                candidate = (phi0 + direction * step, phi1)
            else:
                candidate = (phi0, phi1 + direction * step)
            # *candidate unpacks the tuple into two arguments
            candidate_loss = compute_loss(x, y, *candidate)
            # keep the step if it lowers the loss
            if candidate_loss < best_loss:
                phi0, phi1 = candidate
                best_loss = candidate_loss
                improved = True
                break
        # neither direction helped: halve the step
        if not improved:
            step /= 2

    # best parameters found and their loss
    return phi0, phi1, best_loss

- Run the search: one pass is not enough, because the best intercept
  depends on the slope and vice versa, so we alternate rounds until a
  round no longer lowers the loss.

In [ ]:
# start from the first guess
phi0, phi1 = 0.4, 0.2
loss = compute_loss(x, y, phi0, phi1)
# keep every half-round for the plot below
history = [(0, phi0, phi1, loss)]
for round_num in range(50):
    # tune phi0, then phi1
    phi0, phi1, loss = line_search(x, y, phi0, phi1, vary='phi0')
    history.append((round_num + 0.5, phi0, phi1, loss))
    phi0, phi1, new_loss = line_search(x, y, phi0, phi1, vary='phi1')
    print(f'Round {round_num + 1}: phi0={phi0:.3f}, phi1={phi1:.3f}, loss={new_loss:.3f}')
    history.append((round_num + 1, phi0, phi1, new_loss))
    # stop when a round no longer helps
    if abs(loss - new_loss) < 1e-6:
        break
    loss = new_loss

## Code-along 3. The loss function as a heatmap

- This search descends the loss surface one coordinate at a time
  (**coordinate descent**): an older, calculus-free method, distinct
  from **gradient** descent.

- To visualize it, compute the loss on a grid of (φ0, φ1):

In [ ]:
# a 2D grid of possible phi0 and phi1 values
phi0_mesh, phi1_mesh = np.meshgrid(np.arange(0.0, 2.0, 0.02),
                                   np.arange(-1.0, 1.0, 0.02))
# a 2D array for the losses
all_losses = np.zeros_like(phi1_mesh)
# the loss for every combination of phi0, phi1
for indices, temp in np.ndenumerate(phi1_mesh):
    all_losses[indices] = compute_loss(x, y, phi0_mesh[indices], phi1_mesh[indices])

- Plot the loss as a heatmap with the search path and the best line:

In [ ]:
# loss as a heatmap with gray contour lines
fig = plt.figure()
ax = plt.axes()
fig.set_size_inches(7, 7)
ax.contourf(phi0_mesh, phi1_mesh, all_losses, 256)
ax.contour(phi0_mesh, phi1_mesh, all_losses, 40, colors=['#80808080'])
ax.set_ylim([-1, 1])
ax.set_xlabel(r'Intercept, $\phi_0$')
ax.set_ylabel(r'Slope, $\phi_1$')

# the search path: a staircase, one axis per step
path_phi0 = [h[1] for h in history]
path_phi1 = [h[2] for h in history]
ax.plot(path_phi0, path_phi1, '-', color='white', linewidth=1)

# the best parameters: close to the minimum
ax.plot(phi0, phi1, 'ro')
plt.show()